In [20]:
import sys
import time
import cv2
import json
import numpy as np
from pathlib import Path
from pprint import pprint

# lerobot
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.dataset_tools import add_feature, remove_feature
from lerobot.constants import OBS_ENV_STATE 

# yolo
from ultralytics import YOLO
from yolo_utils import yolo_postprocess_res, yolo_preprocess_tensor

# paths
sys.path.append(str(Path().resolve().parent))
from paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR

# set up env secrets
from dotenv import load_dotenv
load_dotenv(REPO_ROOT/".env", override=True)

# magic autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1.Build existing dataset:

In [21]:
REPO_NAME     = 'so101_car_pick_and_place'
EXPERIMENT_NAME = 'v0' # for the yolo submodel
PUSH_TO_HUB   = True
PULL_FROM_HUB = True # should we update the ds before progressing
DS_SUBFOLDER = 'bboxes'
MODEL_PATH      = POLICIES_DIR / 'yolo' / REPO_NAME / EXPERIMENT_NAME / 'best.pt'

In [29]:
ds = LeRobotDataset(repo_id = f"{HF_NAME}/{REPO_NAME}", root = DATASETS_DIR / REPO_NAME, download_videos = False, video_backend='pyav')
if PULL_FROM_HUB:
    ds.pull_from_repo()

Fetching 463 files: 100%|██████████| 463/463 [00:00<00:00, 1262.73it/s]


In [30]:
if 'observation.environment_state' in ds.features.keys():
    print('Warning - environment state already exists:')
    pprint(ds.features['observation.environment_state'])

Warning - environment state already exists:
{'dtype': 'float32', 'names': ['x_px', 'y_px', 'rotation_deg'], 'shape': (3,)}


Build model:

In [31]:
model = YOLO(MODEL_PATH)

### 2. Build new feature:

In [32]:
new_feature_name = OBS_ENV_STATE
new_feature_meta = {
    "dtype": "float32",
    "names": [
        "source_x",
        "source_y",
        "source_r",
        "target_x",
        "target_y",
        "target_r",
    ],
    "shape":[
        6
    ]
}

### 3. Execute YOLO on the existing dataset

Set up directories:

In [33]:
root = DATASETS_DIR / REPO_NAME / DS_SUBFOLDER
ann_json_dir = root / "annotations"
video_dir    = root / "annotated_videos"

ann_json_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
new_feature_data = np.zeros((len(ds), ), dtype=object)
annotation_data  = []

# scan episodes
for ep, start_idx in enumerate(ds.meta.episodes['dataset_from_index']):
    end_idx = ds.meta.episodes['dataset_to_index'][ep]
    print(f"Episode {ep}: frames [{start_idx}:{end_idx}]")

    writer = None 
    ep_ann = []
    for idx in range(start_idx, end_idx):
        sample = ds[idx]
        frame = sample["observation.images.top_cam"] # torch

        # predict
        rgb = yolo_preprocess_tensor(frame)
        res = model.predict(rgb, conf=0.5, verbose=False)[0] # predict
        vec, ann_img = yolo_postprocess_res(res)   # vec = [sx,sy,sr, tx,ty,tr]

        # store
        new_feature_data[idx] = np.array(vec, dtype=np.float32)
        ep_ann.append({ 
            "frame_index": int(idx),
            "episode": int(ep),
            "vec": [float(v) for v in vec],
        })

        # lazy init writer
        if writer is None:
            W, H = ann_img.width, ann_img.height
            video_path = video_dir / f"ep_{ep:03d}.mp4"
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(video_path), fourcc, 24, (W, H))

        # write frame
        frame_bgr = cv2.cvtColor(np.array(ann_img), cv2.COLOR_RGB2BGR)
        writer.write(frame_bgr)

    # close video
    if writer is not None:
        writer.release()

    # save per-episode JSON
    ep_json_path = ann_json_dir / f"ep_{ep:03d}.json"
    with open(ep_json_path, "w") as f:
        json.dump(ep_ann, f, indent=2)

    # record episode summary
    annotation_data.append({
        "episode": int(ep),
        "frame_range": [int(start_idx), int(end_idx)],
        "json_path": str(ep_json_path),
        "video_path": str(video_path),
    })

# Save global index
with open(ann_json_dir / "index.json", "w") as f:
    json.dump(annotation_data, f, indent=2)

print("YOLO annotation complete.")

Episode 0: frames [0:684]


/home/jonathan/miniconda3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
/home/jonathan/miniconda3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


YOLO annotation complete.


### 4. Add YOLO data as new feature in dataset